# Oefening recommender systems

Installatie package voor deze les: `conda install -c conda-forge implicit` OF `pip install implicit`. Mogelijk zijn er nog packages nodig zoals `pip install h5py`

We vertrekken van simpele voorbeelddata:

In [2]:
import pandas as pd

data = {
    'user_id': [1, 1, 2, 2, 3, 3, 3, 4, 4, 5, 5, 6],
    'song_id': [1, 2, 1, 3, 4, 5, 5, 3, 5, 2, 4, 6],
    'rating': [4, 3, 5, 2, 4, 1, 3, 5, 2, 4, 3, 5]
}

df = pd.DataFrame(data)


Zet de matrix met behulp van een df.pivot_table om naar een werkbaar formaat (rijen: users; kolommen: films)

In [3]:
user_item_matrix = df.pivot_table(index='user_id', columns='song_id', values='rating', fill_value=0)

In [4]:
user_item_matrix.head()

song_id,1,2,3,4,5,6
user_id,,,,,,
1,4,3,0,0,0,0
2,5,0,2,0,0,0
3,0,0,0,4,2,0
4,0,0,5,0,2,0
5,0,4,0,3,0,0


In [5]:

from scipy.sparse import csr_matrix
sparse_user_item_matrix = csr_matrix(user_item_matrix.values)

In [6]:
import implicit

# initialize a model
model = implicit.als.AlternatingLeastSquares(factors=50)

# train the model on a sparse matrix of user/item/confidence weights
model.fit(sparse_user_item_matrix)

user_id = 1
# recommend items for a user
recommendations = model.recommend(user_id, sparse_user_item_matrix[user_id])
print(recommendations)

item_id = 1
# find related items
related = model.similar_items(item_id)
print(related)

/home/bruno/miniconda3/envs/ml_algorithms/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/bruno/miniconda3/envs/ml_algorithms/lib/python3.11/site-packages/implicit/cpu/als.py:95: RuntimeWarning: Intel MKL BLAS is configured to use 6 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'MKL_NUM_THREADS=1' or by callng 'threadpoolctl.threadpool_limits(1, "blas")'. Having MKL use a threadpool can lead to severe performance issues
  check_blas_config()
100%|██████████| 15/15 [00:01<00:00, 12.19it/s]

(array([4, 1, 5, 3, 2, 0, 0, 0, 0, 0], dtype=int32), array([ 3.7071556e-03,  2.0899624e-03,  2.3338944e-06, -8.8975579e-04,
       -3.4028235e+38, -3.4028235e+38,  0.0000000e+00,  0.0000000e+00,
        0.0000000e+00,  0.0000000e+00], dtype=float32))
(array([1, 3, 0, 5, 4, 2, 0, 0, 0, 0], dtype=int32), array([ 1.0000001e+00,  5.7803117e-02,  3.3725832e-02, -1.8946885e-04,
       -3.3015355e-02, -1.1479480e-01,  0.0000000e+00,  0.0000000e+00,
        0.0000000e+00,  0.0000000e+00], dtype=float32))


# uitgewerkt voorbeeld - Muziek aanbevelen met last.fm 360K dataset.

### Dataset binnenhalen
Eerst data binnenhalen:

In [6]:
from implicit.datasets.lastfm import get_lastfm
artists, users, artist_user_plays = get_lastfm()

`artist_user_plays` is een ijle matrix, waarbij elke rij overeenkomt met een andere muzikant en elke kolom overeenkomt met een andere gebruiker. De niet-nulle entries in de `artist_user_plays`-matrix bevatten het aantal keren dat de gebruiker die artiest heeft afgespeeld. De variabelen `artists` en `users` zijn arrays die labels bevatten voor elke rij en kolom in de `artist_user_plays`-matrix.

De `implicit` package richt zich uitsluitend op aanbevelingssystemen voor impliciete feedback, waarbij we positieve voorbeelden krijgen van wat de gebruiker heeft geïnteracteerd, maar niet de bijbehorende negatieve voorbeelden van waar gebruikers niet in geïnteresseerd zijn. In dit voorbeeld worden we het aantal keren getoond dat de gebruiker een artiest in de dataset heeft afgespeeld, en kunnen we afleiden dat een hoog afspeelaantal aangeeft dat de gebruiker een artiest leuk vindt. We kunnen echter niet afleiden dat alleen omdat de gebruiker een band nog nooit heeft afgespeeld, dit betekent dat de gebruiker de band niet leuk vindt.

## Het model trainen

Implicit biedt implementaties van verschillende algoritmen voor impliciete recommender systemen. Voor dit voorbeeld kijken we naar het 'AlternatingLeastSquares'-model. Dit model is bedoeld om een binair target te leren of elke gebruiker interactie heeft gehad met elk item - maar weegt elke binaire interactie met een betrouwbaarheidswaarde van hoe zeker we zijn van deze interactie tussen gebruiker en item. De impliciete implementatie maakt gebruik van de waarden van een ijle matrix om de betrouwbaarheid weer te geven, waarbij de niet-nul waardes aangeven of de gebruiker al dan niet interactie heeft gehad met het item.

In [7]:
from implicit.nearest_neighbours import bm25_weight

# weeg de matrix, om zo sommige users die steeds hetzelfde luisteren in te perken
# en om bepaalde héél populaire liedjes te herschalen
artist_user_plays = bm25_weight(artist_user_plays, K1=100, B=0.8)

In [11]:
print(artist_user_plays)

  (0, 73470)	464.12640081352487
  (0, 97856)	395.4254916528028
  (0, 235382)	917.7576795317125
  (0, 266072)	801.254668853217
  (1, 171865)	479.0537259553822
  (2, 180892)	701.6462524574976
  (3, 285031)	469.2366708878609
  (4, 15103)	274.6530366072618
  (5, 81700)	392.0203057167537
  (6, 284057)	624.2906299439671
  (7, 335320)	482.02241184218633
  (8, 182831)	763.8439378564416
  (9, 12461)	115.27320299060298
  (10, 78717)	79.93913331578632
  (10, 149431)	82.3790196718816
  (10, 220512)	80.7530809045117
  (10, 261830)	81.8842571288024
  (10, 280610)	80.1359339749864
  (10, 297146)	80.33706026790095
  (11, 296825)	764.3099590503471
  (12, 332435)	833.4437619557374
  (13, 41075)	510.69738868947763
  (14, 298571)	759.3034882234267
  (15, 295693)	791.1280662266632
  (16, 185703)	550.3715435013044
  :	:
  (292364, 4775)	178.0367505676167
  (292365, 147943)	970.515999738615
  (292366, 95230)	778.3288671797941
  (292367, 56086)	174.82443977973026
  (292367, 137277)	693.1595624862529
  (292367

In [ ]:
# We hebben (user, item) matrix nodig, en bovendien een matrix in sparse CSR formaat. Zoek in de documentatie naar hulp
user_plays = #TODO

Nu kunnen we trainen, dit neemt wel wat tijd in beslag:

In [ ]:
from implicit.als import AlternatingLeastSquares

# zet het AlternatingLeastSquares algoritme op met parameters factor 64, regularisatie 0.05, en alpha 2.0
# Fit de data.

### Voorspellingen

Nu kunnen er voorspellingen worden gemaakt. Dit kan zowel individueel als in batch.

In [ ]:
# Maak 15 voorspellingen voor user 12345
# Gebruik de .recommend functie, zoek op hoe je deze moet gebruiken

De `.recommend`-functie zal de *N* beste aanbevelingen geven voor elke user in de input, en de itemids teruggeven in de `ids` array, (en ook de berekende scores in de `score` array). Je kunt dus inschatten hoe groot het verschil tussen items was. De muzikanten die worden aanbevolen voor elke user zitten in de ids van de `artist` array:

In [ ]:
# zet het resultaat in een pandas tabel
# maak een kolom voor artiesten (ids), score, en 'already_liked'
# voor dat laatste kan https://numpy.org/doc/stable/reference/generated/numpy.in1d.html handig zijn

De `already_liked` toont of een user al heeft geinterageerd met het item, en in dit voorbeeld is dit voor de meeste items het geval. Je kunt deze reuit filteren met de parameter `filter_already_liked_items`. De `user_plays [userid]`
 parameter is gebruikt om op te zoeken met welke items een user heeft geïnterageerd, je kan dit op `None` zetten als je de voorkeuren van de eigen user niet wil meenemen.

 Er zijn ook opties in de `filter_items` parameter en de `items` parameter, je kan de voorstelling van de user ook on the fly herberekenen met `recalculate_user` parameter.

Meer details kan je online vinden in de documentatie van deze library.

## Gelijkaardige items tonen

Volgende code toont items gelijkaardig aan *The Beatles*:

In [ ]:
# Toon gelijkaardige items voor The Beatles (itemid = 25512)

# Toon het resultaat met deze code hieronder voor een nette tabel
pd.DataFrame({"artist": artists[ids], "score": scores})

### Batch recommendations

De `.recommend`, `.similar_items` en `.similar_users` functies hebben allemaal de mogelijkheid om batches te berekenen. Je kunt een array van userids of itemids meegeven, and dan geeft het systeem een 2D array trug van ids en scores, met in elke rij van de output matrix corresponderend met de waarde van de input. Dit is vooral veel efficiënter, omdat dit in parallel kan worden berekend op grote batches (véél beter dan for loops gebruiken).

In [ ]:
# Geef suggesties voor de eerste 1000 users in dit system (ids tot 1000)